<a href="https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / Scoring** (with a classification model underneath it)

Refresh / Content Opportunity Scoring isn't really "will this page decline: yes/no"
in isolation — the actual decision is *ordering*: given limited review capacity,
which pages go first? That's a ranking problem. The starter pipeline builds this
by training a classifier (probability of decline) and then using that probability
to rank pages, which is a standard way to turn classification into a ranked list.

It's not clustering (I'm not looking for groups of similar pages) and it's not
pure regression (I don't need an exact number, just a good ordering). It's
scoring: assign each page a number that sorts it correctly relative to review
priority.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df)} pages available to rank — confirms this is a ranking-scale problem, not a handful of cases.")

30000 pages available to rank — confirms this is a ranking-scale problem, not a handful of cases.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy target:** `is_declining_label = trend_direction == "down"` (the starter's
label). This comes from a **defined rule**, not a true future observed outcome —
it's a same-window bucket calculated from current trend direction, not "this page
lost traffic over the next 30 days." That's a known weakness I flagged in w01:
it's a beginner proxy, useful for getting the pipeline running, but not the ideal
capstone target.

**Where a stronger label would come from:** an observed future outcome — e.g.
using the warehouse's daily fact table to build `features from prior 90 days ->
decline over next 30 days`. That would be a genuinely observed outcome rather
than a same-window rule, and it's the direction I want to move toward once I use
the full warehouse release instead of just the starter CSV.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

target_counts = df["trend_direction"].value_counts()
print(target_counts)
print(f"\nProxy label distribution: {(df['trend_direction']=='down').mean()*100:.1f}% flagged declining")

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Proxy label distribution: 54.2% flagged declining


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.**

This matches the real decision directly: a reviewer with limited capacity opens
roughly the top 50 ranked pages per cycle, so what matters is how many of those
50 are genuinely worth reviewing — not overall accuracy across all 30,000 pages,
most of which the reviewer will never look at anyway.

'Good' here is beating the transparent baseline rule. The starter pipeline
already shows this is achievable: baseline Precision@50 = 0.240 vs random forest
Precision@50 = 0.740 (verified in `outputs/model_report.md`). Any model I build
should be judged against that same baseline number, on the same kind of
client-holdout split.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Target to beat — baseline rule Precision@50: 0.240")
print("Existing evidence this is achievable — random forest Precision@50: 0.740")
print("Both figures verified from outputs/model_report.md in the starter pipeline.")

Target to beat — baseline rule Precision@50: 0.240
Existing evidence this is achievable — random forest Precision@50: 0.740
Both figures verified from outputs/model_report.md in the starter pipeline.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page (`content_id`). Below is the actual dataframe slice
relevant to this lane — the identifying/join column plus the core signals a
ranking score would be built from.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

lane_columns = ["content_id", "impressions_90d", "sessions_90d", "avg_position",
                 "ctr", "content_age_days", "days_since_last_update",
                 "word_count", "trend_direction"]

lane_slice = df[lane_columns]
print(f"Shape: {lane_slice.shape[0]} rows (pages), {lane_slice.shape[1]} columns")
lane_slice.head(10)


Shape: 30000 rows (pages), 9 columns


,content_id,impressions_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,trend_direction
0,content_304f48230142,3803,17,10.6,0.76,187,20,3221.0,down
1,content_a1fb4e703a9e,15320,9,20.3,0.05,445,25,2481.0,down
2,content_9aa793d4d895,12581,11,36.5,0.09,141,20,3515.0,down
3,content_331d6c4de07b,11751,78,6.2,0.49,463,22,NaN,stable
4,content_d99b7a2d90ca,19140,145,44.0,0.13,263,14,2803.0,down
5,content_d4084a4bc775,3970,5,8.5,0.03,147,20,3080.0,down
6,content_9a34b442b552,20,1,7.0,0.00,90,20,3059.0,down
7,content_a63219c6e95a,1724,28,21.2,0.06,445,22,NaN,stable
8,content_5e6c160719bc,32574,68,46.0,0.09,90,20,3807.0,down
9,content_c27558df2b0c,1240,3,4.9,0.16,257,104,NaN,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The existing hand-rule (`baseline_refresh_score`) is a fixed linear weighting:
`0.40*visibility + 0.30*freshness_risk + 0.25*position_opportunity +
0.05*depth_gap`. That's an if-this-then-that formula with fixed weights — it
can't capture *interactions* between signals. For example, "declining AND high
impressions AND recently ranked page-one" might matter far more together than
each signal weighted and summed separately — a page with all three conditions at
once might deserve a much higher score than the sum of its parts suggests, and a
linear rule structurally can't express that. A tree-based model can split on
combinations of conditions and learn which combinations matter most, which is
exactly the kind of pattern a fixed formula misses.

This isn't a hypothesis — the Week 1-2 results already show it happening: the
same signals, run through a random forest instead of the linear baseline, took
Precision@50 from 0.240 to 0.740 with no new data added, just a model that can
capture interactions.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

numeric_check = df[["impressions_90d", "avg_position", "ctr", "content_age_days"]].copy()
numeric_check["is_declining"] = (df["trend_direction"] == "down").astype(int)

correlations = numeric_check.corr()["is_declining"].drop("is_declining")
print("Individual signal correlations with the declining label:")
print(correlations)
print("\nNo single signal is strongly correlated alone — consistent with the pattern")
print("being about combinations of signals, which favors a model over a fixed linear rule.")

Individual signal correlations with the declining label:
impressions_90d    -0.018175
avg_position       -0.029035
ctr                -0.061911
content_age_days   -0.163882
Name: is_declining, dtype: float64

No single signal is strongly correlated alone — consistent with the pattern
being about combinations of signals, which favors a model over a fixed linear rule.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.